# Module 13 · Descriptive Statistics and Regression Analysis

*BUSN 3030E — Python Programming Skills for Business Analytics I*

**Learning objectives.** After this module you will be able to

- summarize numeric and categorical variables with descriptive statistics;
- compare groups with `groupby` and measure association with correlation;
- fit and interpret a linear regression with `statsmodels`, including categorical predictors;
- fit and interpret a logistic regression for a yes/no outcome;
- use a fitted model to make predictions.

## 1 · Preparation

### 1.1 The data

A US health insurer wants to understand what drives the cost of its policies. The data set contains 1,338 policyholders with their age, sex, body mass index (BMI), number of children covered, smoking status, region, and annual medical charges billed (in dollars). *(Source: [Kaggle, US Health Insurance Dataset](https://www.kaggle.com/datasets/teertha/ushealthinsurancedataset).)*

In [1]:
import pandas as pd
import statsmodels.formula.api as smf

DATA_URL = "https://raw.githubusercontent.com/hanisaf/python-programming-skills/main/data/"

insurance = pd.read_csv(DATA_URL + "insurance.csv")
insurance.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


### 1.2 Describing numeric variables

`describe()` computes the standard summary statistics for every numeric column:

In [2]:
insurance.describe().round(2)

,age,bmi,children,charges
count,1338.00,1338.00,1338.00,1338.00
mean,39.21,30.66,1.09,13270.42
std,14.05,6.10,1.21,12110.01
min,18.00,15.96,0.00,1121.87
25%,27.00,26.30,0.00,4740.29
50%,39.00,30.40,1.00,9382.03
75%,51.00,34.69,2.00,16639.91
max,64.00,53.13,5.00,63770.43


| Row | Meaning |
|---|---|
| `count` | number of non-missing values |
| `mean`, `std` | average and standard deviation |
| `min`, `max` | smallest and largest value |
| `25%`, `50%`, `75%` | quartiles; the 50% quantile is the median |

The mean charge (\$13,270) is far above the median (\$9,382): a few very expensive policies pull the mean up. When a distribution is skewed like this, the median is the better description of a "typical" value.

Each statistic is also available as a method, for a single column or for the whole DataFrame:

In [3]:
charges = insurance["charges"]
print(f"mean   {charges.mean():>10,.2f}")
print(f"median {charges.median():>10,.2f}")
print(f"std    {charges.std():>10,.2f}")
print(f"90th percentile {charges.quantile(0.9):,.2f}")

mean    13,270.42
median   9,382.03
std     12,110.01
90th percentile 34,831.72


> **Note:** `insurance.mean()` raises an error because the DataFrame contains text columns. Select the numeric columns first, or use `insurance.mean(numeric_only=True)`.

### 1.3 Describing categorical variables

For text columns, `describe()` reports the number of distinct values and the most frequent one; `exclude="number"` selects the non-numeric columns. `value_counts()` gives the full frequency table.

In [4]:
insurance.describe(exclude="number")

,sex,smoker,region
count,1338,1338,1338
unique,2,2,4
top,male,no,southeast
freq,676,1064,364


In [5]:
insurance["smoker"].value_counts()

smoker
no     1064
yes     274
Name: count, dtype: int64

In [6]:
insurance["region"].value_counts(normalize=True).round(3)   # proportions instead of counts

region
southeast    0.272
southwest    0.243
northwest    0.243
northeast    0.242
Name: proportion, dtype: float64

### 1.4 Comparing groups

`groupby()` splits the rows into groups, applies a statistic to each group, and combines the results. Do smokers pay more?

In [7]:
insurance.groupby("smoker")["charges"].mean().round(2)

smoker
no      8434.27
yes    32050.23
Name: charges, dtype: float64

`agg()` computes several statistics at once, and grouping by two columns gives a cross-tabulation:

In [8]:
insurance.groupby("smoker")["charges"].agg(["count", "mean", "median"]).round(2)

,count,mean,median
smoker,,,
no,1064,8434.27,7345.41
yes,274,32050.23,34456.35


In [9]:
insurance.groupby(["region", "smoker"])["charges"].mean().unstack().round(0)

smoker,no,yes
region,,
northeast,9166.0,29674.0
northwest,8556.0,30192.0
southeast,8032.0,34845.0
southwest,8019.0,32269.0


`unstack()` turns the inner grouping level (`smoker`) into columns, which makes the table easier to read. In every region, smokers are charged three to four times as much as non-smokers.

### 1.5 Correlation

The **correlation coefficient** $r$ measures the strength of the *linear* association between two numeric variables, from −1 (perfect negative) through 0 (none) to +1 (perfect positive). `corr()` computes it for every pair of numeric columns:

In [10]:
insurance.corr(numeric_only=True).round(3)

,age,bmi,children,charges
age,1.000,0.109,0.042,0.299
bmi,0.109,1.000,0.013,0.198
children,0.042,0.013,1.000,0.068
charges,0.299,0.198,0.068,1.000


Charges increase with age ($r = 0.30$) and BMI ($r = 0.20$). The default is Pearson's correlation; `corr(method="spearman")` computes a rank-based version that is less sensitive to outliers and to curved relationships.

Correlation describes association between two variables at a time. To estimate the effect of several variables together, we need regression.

### 1.6 Linear regression

A linear regression models an outcome $y$ as a weighted sum of predictors:

$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \varepsilon$$

The coefficients $\beta$ are estimated by **ordinary least squares (OLS)**, which chooses the values that minimize the squared prediction errors. The `statsmodels` formula interface lets us write the model like the equation: `"outcome ~ predictor1 + predictor2"`.

Fitting a model takes three steps: describe it, fit it, and inspect the results.

In [11]:
model = smf.ols("charges ~ age", data=insurance)   # 1. describe
results = model.fit()                                # 2. fit
print(results.summary())                             # 3. inspect

                            OLS Regression Results                            
Dep. Variable:                charges   R-squared:                       0.089
Model:                            OLS   Adj. R-squared:                  0.089
Method:                 Least Squares   F-statistic:                     131.2
Date:                Tue, 13 Oct 2026   Prob (F-statistic):           4.89e-29
Time:                        16:48:52   Log-Likelihood:                -14415.
No. Observations:                1338   AIC:                         2.883e+04
Df Residuals:                    1336   BIC:                         2.884e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   3165.8850    937.149      3.378      0.0

The summary is long, but four numbers matter most:

| Quantity | Value | Interpretation |
|---|---|---|
| `coef` of `age` | 257.7 | each additional year of age is associated with \$258 more in annual charges |
| `Intercept` | 3,165.9 | the predicted charge at age 0, a mathematical anchor rather than a meaningful value here |
| `P>\|t\|` of `age` | 0.000 | the probability of seeing an effect this strong if age had no effect; below 0.05 we call the effect *statistically significant* |
| `R-squared` | 0.089 | age alone explains 8.9% of the variation in charges |

The same numbers are available as attributes:

In [12]:
print(results.params.round(2))
print("R-squared:", round(results.rsquared, 3))

Intercept    3165.89
age           257.72
dtype: float64
R-squared: 0.089


### 1.7 Multiple regression with categorical predictors

Adding predictors is as simple as extending the formula. Text columns such as `smoker` and `region` are converted automatically into **dummy (indicator) variables**: one category serves as the baseline, and each other category gets its own coefficient.

In [13]:
full = smf.ols("charges ~ age + bmi + children + smoker + sex + region", data=insurance).fit()
full.summary().tables[1]

,coef,std err,t,P>|t|,[0.025,0.975]
Intercept,-1.194e+04,987.819,-12.086,0.000,-1.39e+04,-1e+04
smoker[T.yes],2.385e+04,413.153,57.723,0.000,2.3e+04,2.47e+04
sex[T.male],-131.3144,332.945,-0.394,0.693,-784.470,521.842
region[T.northwest],-352.9639,476.276,-0.741,0.459,-1287.298,581.370
region[T.southeast],-1035.0220,478.692,-2.162,0.031,-1974.097,-95.947
region[T.southwest],-960.0510,477.933,-2.009,0.045,-1897.636,-22.466
age,256.8564,11.899,21.587,0.000,233.514,280.199
bmi,339.1935,28.599,11.860,0.000,283.088,395.298
children,475.5005,137.804,3.451,0.001,205.163,745.838


`summary().tables[1]` displays only the coefficient table. Reading it:

- `smoker[T.yes]` = 23,848: holding the other variables constant, smokers are charged about \$23,800 more than non-smokers (the baseline category, `no`).
- `age` = 257 and `bmi` = 339: each additional year of age adds about \$257, each BMI point about \$339.
- `sex[T.male]` is small and not significant ($p = 0.70$): once smoking, age, and BMI are accounted for, sex makes no detectable difference.
- The region coefficients compare each region with the baseline, `northeast`.

In [14]:
print("R-squared:", round(full.rsquared, 3))

R-squared: 0.751


Together the predictors explain 75% of the variation in charges, far more than age alone.

### 1.8 Predictions

A fitted model can predict the outcome for new cases. Provide the predictors as a DataFrame with the same column names:

In [15]:
applicants = pd.DataFrame({
    "age": [30, 30],
    "bmi": [27.0, 27.0],
    "children": [1, 1],
    "smoker": ["no", "yes"],
    "sex": ["female", "female"],
    "region": ["southeast", "southeast"],
})
full.predict(applicants).round(2)

0     4365.85
1    28214.39
dtype: float64

The two applicants differ only in smoking status, and the predictions differ by exactly the smoker coefficient.

## 2 · Warmup

The diabetes data set comes from the US National Institute of Diabetes and Digestive and Kidney Diseases. It records diagnostic measurements for 768 women of Pima Indian heritage, aged 21 or older, and whether they were diagnosed with diabetes (*Outcome* = 1). *(Source: [Kaggle, Pima Indians Diabetes Database](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database).)*

| Variable | Description |
|---|---|
| `Pregnancies` | number of times pregnant |
| `Glucose` | plasma glucose concentration two hours after an oral glucose tolerance test |
| `BloodPressure` | diastolic blood pressure (mm Hg) |
| `SkinThickness` | triceps skin fold thickness (mm) |
| `Insulin` | two-hour serum insulin (μU/ml) |
| `BMI` | body mass index (kg/m²) |
| `DiabetesPedigreeFunction` | a score of diabetes history in relatives |
| `Age` | age in years |
| `Outcome` | 1 = diabetes, 0 = no diabetes |

In [16]:
diabetes = pd.read_csv(DATA_URL + "diabetes.csv")
diabetes.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


**Exercise 2.1.** Produce the descriptive statistics. Do you notice anything suspicious?

In [17]:
diabetes.describe().round(2)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.00,768.00,768.00,768.00,768.00,768.00,768.00,768.00,768.00
mean,3.85,120.89,69.11,20.54,79.80,31.99,0.47,33.24,0.35
std,3.37,31.97,19.36,15.95,115.24,7.88,0.33,11.76,0.48
min,0.00,0.00,0.00,0.00,0.00,0.00,0.08,21.00,0.00
25%,1.00,99.00,62.00,0.00,0.00,27.30,0.24,24.00,0.00
50%,3.00,117.00,72.00,23.00,30.50,32.00,0.37,29.00,0.00
75%,6.00,140.25,80.00,32.00,127.25,36.60,0.63,41.00,1.00
max,17.00,199.00,122.00,99.00,846.00,67.10,2.42,81.00,1.00


The minimum of `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI` is 0, which is impossible for a living person. In this data set, **0 was used to record a missing measurement**. How many such values are there?

In [18]:
(diabetes[["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]] == 0).sum()

Glucose            5
BloodPressure     35
SkinThickness    227
Insulin          374
BMI               11
dtype: int64

Almost half of the insulin values are missing. Statistics computed on such columns are misleading unless the zeros are treated as missing values. This is why we always describe data *before* modeling it.

**Exercise 2.2.** What share of the women in the data set have diabetes?

In [19]:
diabetes["Outcome"].value_counts(normalize=True).round(3)

Outcome
0    0.651
1    0.349
Name: proportion, dtype: float64

**Exercise 2.3.** Compare the average glucose, BMI, and age of women with and without diabetes.

In [20]:
diabetes.groupby("Outcome")[["Glucose", "BMI", "Age"]].mean().round(1)

,Glucose,BMI,Age
Outcome,,,
0,110.0,30.3,31.2
1,141.3,35.1,37.1


Women with diabetes have markedly higher glucose levels, and are on average heavier and older.

**Exercise 2.4.** Which variable is most strongly correlated with the outcome?

In [21]:
diabetes.corr()["Outcome"].drop("Outcome").sort_values(ascending=False).round(3)

Glucose                     0.467
BMI                         0.293
Age                         0.238
Pregnancies                 0.222
DiabetesPedigreeFunction    0.174
Insulin                     0.131
SkinThickness               0.075
BloodPressure               0.065
Name: Outcome, dtype: float64

Selecting the `Outcome` column of the correlation matrix shows each variable's correlation with the outcome; `drop()` removes the trivial correlation of `Outcome` with itself. Glucose stands out.

## 3 · Workout

**Exercise 3.1.** Model the probability of diabetes as a function of the diagnostic measurements.

The outcome is binary (0 or 1), so linear regression is a poor fit: it can predict values below 0 or above 1. **Logistic regression** instead models the *probability* of the outcome, which always lies between 0 and 1, through an S-shaped curve:

$$P(y = 1) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + \dots)}}$$

In `statsmodels`, replace `ols` with `logit`. We use the predictors that have few missing values.

In [22]:
logit = smf.logit("Outcome ~ Pregnancies + Glucose + BMI + DiabetesPedigreeFunction + Age",
                  data=diabetes).fit(disp=False)
logit.summary().tables[1]

,coef,std err,z,P>|z|,[0.025,0.975]
Intercept,-8.6731,0.690,-12.570,0.000,-10.025,-7.321
Pregnancies,0.1195,0.032,3.774,0.000,0.057,0.181
Glucose,0.0329,0.003,9.667,0.000,0.026,0.040
BMI,0.0795,0.014,5.760,0.000,0.052,0.107
DiabetesPedigreeFunction,0.8915,0.292,3.051,0.002,0.319,1.464
Age,0.0122,0.009,1.346,0.178,-0.006,0.030


Logistic coefficients are on the *log-odds* scale, which is hard to interpret. Exponentiating them gives **odds ratios**: the factor by which the odds of diabetes are multiplied for a one-unit increase in the predictor.

In [23]:
import numpy as np

np.exp(logit.params).round(3)

Intercept                   0.000
Pregnancies                 1.127
Glucose                     1.033
BMI                         1.083
DiabetesPedigreeFunction    2.439
Age                         1.012
dtype: float64

- Each additional point of glucose multiplies the odds of diabetes by 1.033, i.e. raises them by about 3.3%. For a 30-point difference, the odds are multiplied by $1.033^{30} \approx 2.7$.
- Each BMI point raises the odds by about 8.3%; each pregnancy by about 12.7%.
- Pregnancies, glucose, BMI, and the pedigree score are statistically significant at the 5% level. Age is not ($p = 0.18$): once the number of pregnancies is in the model, age adds little information, because older women have had more pregnancies on average.

`fit(disp=False)` silences the progress messages that the fitting algorithm prints otherwise.

**Exercise 3.2.** Estimate the probability of diabetes for two 40-year-old women with two pregnancies, a BMI of 30, and an average pedigree score (0.47), one with a glucose level of 100 and one with 160.

In [24]:
patients = pd.DataFrame({
    "Pregnancies": [2, 2],
    "Glucose": [100, 160],
    "BMI": [30, 30],
    "DiabetesPedigreeFunction": [0.47, 0.47],
    "Age": [40, 40],
})
logit.predict(patients).round(3)

0    0.136
1    0.531
dtype: float64

The model estimates the probability of diabetes at about 14% for the first patient and 53% for the second. For a logistic model, `predict()` returns probabilities.

## Summary

| Task | Code |
|---|---|
| Summary statistics | `df.describe()`, `df.describe(exclude="number")` |
| Single statistics | `s.mean()`, `s.median()`, `s.std()`, `s.quantile(q)` |
| Frequencies | `s.value_counts(normalize=True)` |
| Group comparison | `df.groupby("group")["value"].agg(["mean", "median"])` |
| Correlation | `df.corr(numeric_only=True)` |
| Linear regression | `smf.ols("y ~ x1 + x2", data=df).fit()` |
| Logistic regression | `smf.logit("y ~ x1 + x2", data=df).fit()` |
| Results | `.summary()`, `.params`, `.pvalues`, `.rsquared`, `.predict(new_df)` |